In [14]:
from neo4j_queries import *

In [37]:
import os
from dotenv import load_dotenv, find_dotenv
from neo4j import GraphDatabase

load_dotenv(find_dotenv(usecwd=True), override=True)

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [38]:
import pandas as pd
from pathlib import Path

project_root = Path(find_dotenv(usecwd=True)).parent
df = pd.read_csv(project_root / 'dataset/03_cleaned/cleaned_restaurant.csv')

df.head()

,url,name,category,rating,reviews,price,address,status,snippet,img,postal,lon,lat,nearest_mrt,name_norm,address_norm,category_clean,category_group
0,https://www.google.com/maps/place/$1.50+Dim+Su...,$1.50 Dim Sum Bedok,Hawker Stall,3.7,(479),$1.50,"Block 123 Bedok North Street 2, 01-142",Open 24 hours,"""Very reasonable price n very delicious""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,460123.0,103.937343,1.329199,"['bedok_north', 'bedok_reservoir']",$1.50 DIM SUM BEDOK,"BLOCK 123 BEDOK NORTH STREET 2, 01-142",hawker stall,hawker
1,https://www.google.com/maps/place/%2301-12+Han...,#01-12 Handmade Noodles,Hawker Stall,4.4,(12),$1–10,"4A Eunos Cres, #01-99 4A",Hawker Stall,"""It tastes not bad at all, extra crispy one si...",https://lh3.googleusercontent.com/gps-cs-s/AHV...,402004.0,103.904256,1.320331,['eunos'],#01-12 HANDMADE NOODLES,"4A EUNOS CRES, #01-99 4A",hawker stall,hawker
2,https://www.google.com/maps/place/%2301-18+%E5...,#01-18 廣東砂煲飯。小吃,Hawker Stall,4.0,(5),$1–10,"#01-18 Geylang Bahru, Market & Food Centre",Hawker Stall,"""Very authentic and good claypot rice.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,330069.0,103.870005,1.321463,['geylang_bahru'],#01-18 廣東砂煲飯。小吃,"#01-18 GEYLANG BAHRU, MARKET & FOOD CENTRE",hawker stall,hawker
3,https://www.google.com/maps/place/%2301-20+%E8...,#01-20 萝卜糕 经济小吃,Hawker Stall,4.7,(11),$1–10,"22A Havelock Rd, #01-20",Closed · Opens 8 am,"""Fried till crispy yet not greasy.""",https://lh3.googleusercontent.com/gps-cs-s/AHV...,161022.0,103.829623,1.287971,['havelock'],#01-20 萝卜糕 经济小吃,"22A HAVELOCK RD, #01-20",hawker stall,hawker
4,https://www.google.com/maps/place/%2301-35+%E5...,#01-35 健康齋素食,Hawker Stall,5.0,(1),NaN,"79 Telok Blangah Dr, #01-35, Telok Blangah Foo...",Closed · Opens 6 am,Takeaway,NaN,100079.0,103.807618,1.273356,['telok_blangah'],#01-35 健康齋素食,"79 TELOK BLANGAH DR, #01-35, TELOK BLANGAH FOO...",hawker stall,hawker


In [44]:
def create_constraints(tx):
    tx.run("CREATE CONSTRAINT restaurant_name IF NOT EXISTS FOR (r:Restaurant) REQUIRE r.name IS UNIQUE")
    tx.run("CREATE CONSTRAINT mrt_name IF NOT EXISTS FOR (m:MRT) REQUIRE m.name IS UNIQUE")
    tx.run("CREATE CONSTRAINT category_name IF NOT EXISTS FOR (c:FoodCategory) REQUIRE c.name IS UNIQUE")

with driver.session() as session:
    session.execute_write(create_constraints)

In [45]:
def insert_batch(tx, rows):
    tx.run("""
        UNWIND $rows AS row
        MERGE (r:Restaurant {name: row.name})
        ON CREATE SET
            r.rating = row.rating,
            r.reviews = row.reviews,
            r.address = row.address,
            r.lat = row.lat,
            r.lon = row.lon,
            r.price = row.price

        WITH r, row
        UNWIND row.mrts AS mrt_name
        MERGE (m:MRT {name: mrt_name})
        MERGE (r)-[:IS_NEAR_TO]->(m)

        WITH r, row
        MERGE (c:FoodCategory {name: row.category})
        MERGE (r)-[:FOOD_CATEGORIZED_AS]->(c)
    """, rows=rows)

In [46]:
import ast

# Clear existing data first
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")
    print("Cleared existing data.")

BATCH_SIZE = 500

records = df[["name", "rating", "reviews", "address", "lat", "lon", "price", "nearest_mrt", "category_clean"]].copy()
records["mrts"] = records["nearest_mrt"].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
records = records.rename(columns={"category_clean": "category"}).drop(columns=["nearest_mrt"]).to_dict("records")

with driver.session() as session:
    for i in range(0, len(records), BATCH_SIZE):
        batch = records[i:i + BATCH_SIZE]
        session.execute_write(insert_batch, batch)
        print(f"Inserted {min(i + BATCH_SIZE, len(records))}/{len(records)}")

Cleared existing data.
Inserted 500/8375
Inserted 1000/8375
Inserted 1500/8375
Inserted 2000/8375
Inserted 2500/8375
Inserted 3000/8375
Inserted 3500/8375
Inserted 4000/8375
Inserted 4500/8375
Inserted 5000/8375
Inserted 5500/8375
Inserted 6000/8375
Inserted 6500/8375
Inserted 7000/8375
Inserted 7500/8375
Inserted 8000/8375
Inserted 8375/8375


In [57]:
search_restaurants(driver, "eunos", "hawker stall")

[{'name': '州记鸡饭烧腊', 'rating': 4.6, 'address': '29 Eunos Ave 6'},
 {'name': '福春叻沙卤面 (Fu Chun Laksa & Lor Mee)',
  'rating': 4.4,
  'address': '4A Eunos Cres, #01-25'},
 {'name': '鼎记 Ding Ji Eunos', 'rating': 1.9, 'address': '2A Eunos Cres'},
 {'name': 'AL-AMEEN',
  'rating': 3.1,
  'address': 'Blk 4A Eunos Cres, #01-22 Market & Food Centre'},
 {'name': 'ALHAM EUNOS (HAWKER CENTRE)',
  'rating': 4.5,
  'address': '4A Eunos Cres'},
 {'name': 'AROMA NASI LEMAK',
  'rating': 4.3,
  'address': '4A Eunos Cres, #01-08'},
 {'name': 'Chokdee Beancakes House',
  'rating': 5.0,
  'address': '4A Eunos Cres, #01-66/67 Market & Food Centre'},
 {'name': 'Duck Fong Hong Kong Style Roasted Delight',
  'rating': 2.1,
  'address': '2A Eunos Cres'},
 {'name': 'Eng Kee Hainanese Chicken Rice & Porridge',
  'rating': 4.3,
  'address': '4A Eunos Cres, #01-34 Market and Food Centre'},
 {'name': 'Epok Epok Central',
  'rating': 3.9,
  'address': '4A Eunos Cres, #01-09 Market & Food Centre'},
 {'name': 'Eunos MR

In [49]:
get_graph_summary(driver)

{'total_nodes': 8445,
 'restaurants': 8009,
 'mrt_stations': 141,
 'categories': 295,
 'relationships': [{'type': 'IS_NEAR_TO', 'count': 12209},
  {'type': 'FOOD_CATEGORIZED_AS', 'count': 8120}]}

In [51]:
# preview_nodes(driver, "Restaurant", 5)
preview_nodes(driver, "MRT", 5)
# preview_nodes(driver, "FoodCategory", 5)

[<Node element_id='4:457459c0-8178-43e7-b9be-f24d7bd5bc95:99' labels=frozenset({'MRT'}) properties={'name': 'bedok_north'}>,
 <Node element_id='4:457459c0-8178-43e7-b9be-f24d7bd5bc95:100' labels=frozenset({'MRT'}) properties={'name': 'bedok_reservoir'}>,
 <Node element_id='4:457459c0-8178-43e7-b9be-f24d7bd5bc95:101' labels=frozenset({'MRT'}) properties={'name': 'eunos'}>,
 <Node element_id='4:457459c0-8178-43e7-b9be-f24d7bd5bc95:102' labels=frozenset({'MRT'}) properties={'name': 'geylang_bahru'}>,
 <Node element_id='4:457459c0-8178-43e7-b9be-f24d7bd5bc95:103' labels=frozenset({'MRT'}) properties={'name': 'havelock'}>]

In [52]:
check_isolated_restaurants(driver)

[]